# Microglia motility — analysis tutorial (Step 8–9)

**What this notebook is.** A walk-through of the Python half of the pipeline: it takes the two aggregated TrackMate CSVs (`all_spots.csv`, `all_tracks.csv`) and turns them into the count and motility figures, then adds two new analyses — **mean squared displacement (MSD)** and **per-image trajectory maps**.

**How to read it.** Every code cell has a markdown cell above it explaining *what* it does, *why* it's done that way, and the specific pandas / matplotlib moves used — so you can rebuild it yourself. Comments inside the code explain the lines; the markdown explains the ideas.

---

### The data model (read this once)

Each **spot** = one microglia detection in one frame. Each **track** = one cell followed across frames (a chain of spots sharing a `TRACK_ID`). Two files:

| File | One row = | Key columns we use |
|------|-----------|--------------------|
| `all_spots.csv` | one detection in one frame | `POSITION_X/Y`, `FRAME`, `POSITION_T`, `MEDIAN_INTENSITY_CH5` (region), `CIRCULARITY`, `TRACK_ID` |
| `all_tracks.csv` | one whole track | `NUMBER_SPOTS`, `NUMBER_GAPS`, `CONFINEMENT_RATIO`, `MEAN_STRAIGHT_LINE_SPEED`, `TRACK_MEDIAN_SPEED` |

The region label is carried **per spot** in `MEDIAN_INTENSITY_CH5` — the value of channel 5 (the region map your Step 04 macro paints) sampled under each cell. Verified against your current macro and sample data, the values are:

| CH5 value | Region | Notes |
|-----------|--------|-------|
| `0` | outside annotated tissue | non–spinal-cord cells; usually excluded |
| `1` | uninjured side | |
| `2` | injured side (penumbra) | |
| `3` | injury core | the tight ROI inside the injured side |

> **Two granularities, used deliberately.** For *counts* we collapse to **injured vs uninjured** (core folds into injured, because it *is* the injured side and splitting it would undercount). For *motility* we keep the full **core / penumbra / uninjured** gradient, because the whole question is whether cells behave differently as you move toward the core.

---

### Before you trust the numbers — three judgement calls baked in

1. **Why median, not mean, for the region label.** A cell's footprint can straddle a region boundary. `MEAN_INTENSITY_CH5` then comes back fractional (e.g. 1.7) and is meaningless. `MEDIAN_INTENSITY_CH5` returns the value covering *most* of the cell — a clean integer. Always use the median for label-map channels. (Your sample data confirms it: the mean column is full of fractions like 0.28; the median column is exactly 0/1/2/3.)
2. **Unlinked detections.** ~11% of spots in the sample have no `TRACK_ID` (TrackMate detected them but couldn't link them into a track). They are real cells for *counting* but cannot contribute to *motility*. Cell 3 reports how many; Cell 4 lets you choose whether counts include them.
3. **Cadence differs between images** (30 vs 40 min/frame here). So "frame 5" is a different real time in different fish. Every time axis below is converted to **minutes**, never left in frames.


## Cell 1 — Load the two CSVs into the session

In Colab, `files.upload()` opens a browser dialog and writes whatever you pick into `/content/`. You can select **both** CSVs at once. The returned `uploaded` dict maps filename → bytes; we only need the keys (the filenames), which we use in the next cell.

If you're running locally instead of Colab, skip this cell and just set the two paths by hand in Cell 2.

In [ ]:
# Run this, then pick BOTH all_spots.csv AND all_tracks.csv in the dialog.
from google.colab import files
uploaded = files.upload()                 # browser dialog; multi-select is allowed
print("\nUploaded:", list(uploaded.keys()))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def export_button(df, filename, label=None):
    """A one-click 'download this plot's data' button.
    Writes df to CSV and (in Colab) triggers a browser download."""
    btn = widgets.Button(description=label or f"⬇  Export  {filename}",
                         layout=widgets.Layout(width="auto"))
    out = widgets.Output()
    def _click(_):
        with out:
            out.clear_output()
            df.to_csv(filename, index=False)
            print(f"wrote {filename}  ({len(df)} rows × {df.shape[1]} cols)")
            try:
                from google.colab import files     # Colab -> browser download
                files.download(filename)
            except Exception:
                print("(not in Colab — saved next to the notebook instead)")
    btn.on_click(_click)
    display(btn, out)

## Cell 2 — Read them into DataFrames (the 4-row header trick)

TrackMate writes **four header rows**, not one:

```
row 0:  ALL_CAPS machine keys    <- the real column names we want
row 1:  Human readable names     \
row 2:  Short abbreviations       }  decoration we must throw away
row 3:  Units                    /
```

`pd.read_csv` applies `skiprows` **first**, then reads `header` from what's left. So:
- `header=0` → use row 0 (the keys) as column names
- `skiprows=[1,2,3]` → drop the three decoration rows so they aren't read as data

We find which file is which **by name** (`"spots"`/`"tracks"` in the filename) so upload order doesn't matter. Note: these files were concatenated from 9 source images by your Step 07 export, which prepends a `SOURCE_IMAGE` column — that's the one non-stock column.

In [ ]:
import pandas as pd

# If you're NOT in Colab, replace the next two lines with literal paths, e.g.:
#   spots_file, tracks_file = "all_spots.csv", "all_tracks.csv"
spots_file  = next(f for f in uploaded if "spots"  in f.lower())
tracks_file = next(f for f in uploaded if "tracks" in f.lower())

# The 4-row-header read pattern — memorise this; you'll use it for every TrackMate CSV.
spots  = pd.read_csv(spots_file,  header=0, skiprows=[1, 2, 3])
tracks = pd.read_csv(tracks_file, header=0, skiprows=[1, 2, 3])

print(f"spots  : {spots.shape[0]:>5} rows x {spots.shape[1]} cols")
print(f"tracks : {tracks.shape[0]:>5} rows x {tracks.shape[1]} cols")
print(f"source images: {spots['SOURCE_IMAGE'].nunique()}")

## Cell 3 — Integrity checks (look before you leap)

No analysis here — this cell only *verifies the assumptions the rest of the notebook depends on* and adds one derived column (`is_manual`). Read all five printouts before trusting anything downstream. New techniques introduced:

- **`value_counts(dropna=False)`** — frequency table that also counts missing values, so nothing hides.
- **`.groupby([...]).ngroups`** — counts distinct *combinations* of keys. We use it to show why `TRACK_ID` alone is not a cell identity.
- **A boolean column from a comparison** (`spots["CIRCULARITY"] == 1.0`) — vectorised, no loop.

**Why `(SOURCE_IMAGE, TRACK_ID)` is the true cell key.** Each image restarts `TRACK_ID` from 0, so track 4 in fish A and track 4 in fish B are different cells that would collide if you grouped on `TRACK_ID` alone. We group on **both** columns everywhere.

**The `is_manual` flag.** Spots you placed by hand during curation are perfect circles (`CIRCULARITY == 1.0` exactly, radius 5). Automated MaskDetector spots inherit the real cell outline, so circularity < 1. We *flag* rather than drop them: a hand-placed spot is a real cell (fine for counting and for position-based motility) but its *shape* is a placeholder (exclude it from any morphology metric).

**The unlinked-detection report (item 5)** is the new one — it surfaces the spots with no `TRACK_ID` so you can see how many there are before deciding what counts should include.

In [ ]:
import pandas as pd
from IPython.display import display   # display() renders DataFrames as HTML tables in Colab,
                                      # unlike print(), which falls back to plain monospace text.

# ----------------------------------------------------------------------------
# Lookup tables / small helpers defined ONCE at the top.
# Keeping these out of the logic below makes each section read like prose.
# ----------------------------------------------------------------------------

# The region code is stored as a float (0.0–3.0), not text, so the dict keys
# must also be floats — 0 and 0.0 are equal in Python, but being explicit here
# documents the data type and avoids surprises if you ever switch to .map().
REGION_NAMES = {
    0.0: "outside tissue",
    1.0: "uninjured",
    2.0: "injured (penumbra)",
    3.0: "injury core",
}

# Pull a short, UNIQUE label out of the 80-char SOURCE_IMAGE string.
# Filenames look like  YYYYMMDD_YYYYMMDD_PositionXXX_..._corrected_xyz
# so parts[0][4:] = 'MMDD' (drops the year) and parts[2] = 'PositionXXX'.
# Joining them ('0724·Position008') keeps fish from different imaging dates
# distinct — Position008 is reused across sessions, exactly like TRACK_ID is [1].
def short(s):
    parts = s.split("_")
    return f"{parts[0][4:]}·{parts[2]}"   # '0724·Position008'
def short(s):
    parts = s.split("_")
    return f"{parts[0][4:]}·{parts[2]}"   # '0724·Position008'



def banner(txt):
    """Print a short underlined header so the output is scannable.
    len(txt) makes the underline exactly as wide as the title."""
    print(f"\n{txt}\n" + "-" * len(txt))


# ============================================================================
# 1. REGION LABEL COUNTS  (codes -> human-readable names, with percentages)
# ============================================================================
banner("1) Region label counts  (MEDIAN_INTENSITY_CH5)")

# value_counts() tallies how many spots carry each code.
#   dropna=False  -> also count rows where the label is missing (NaN),
#                    so the totals can never silently disagree with len(spots).
#   sort_index()  -> order by the code (0,1,2,3) instead of by frequency,
#                    so the table always reads outside -> uninjured -> ... -> core.
rc = (spots["MEDIAN_INTENSITY_CH5"]
        .value_counts(dropna=False)
        .sort_index())

# Build a tidy DataFrame for display. rc.index holds the codes; rc.values the counts.
#   .get(k, "??") looks up the name but falls back to "??" for any unexpected code,
#                 so a stray value shows up loudly instead of crashing.
region_tbl = pd.DataFrame(
    {
        "region":  [REGION_NAMES.get(k, "??") for k in rc.index],
        "n_spots": rc.values,
        # element-wise percentage; .round(1) keeps one decimal place.
        "pct":     (100 * rc.values / len(spots)).round(1),
    },
    index=rc.index.rename("code"),   # name the index column 'code' in the rendered table
)
display(region_tbl)   # <- HTML table, not text


# ============================================================================
# 2. MANUAL-CURATION FLAG  (a hand-placed spot has perfect circularity = 1.0)
# ============================================================================
# The "==" comparison returns a boolean Series (True/False per row); assigning it
# back to spots adds a new column WITHOUT dropping anything — we flag, not filter,
# so these spots still count later.
spots["is_manual"] = spots["CIRCULARITY"] == 1.0

# A boolean Series sums as 1/0, so .sum() = number of True values.
# int(...) just converts numpy's int64 to a plain Python int for clean printing.
n_manual = int(spots["is_manual"].sum())

banner("2) Manual (hand-placed) spots")
print(f"{n_manual} / {len(spots)}  "
      f"({100 * n_manual / len(spots):.1f}%)  — flagged, not dropped")


# ============================================================================
# 3. TRACK IDENTITY CHECK
#    A TRACK_ID is only unique WITHIN one image. Two different images can both
#    contain a "track 0", so TRACK_ID alone under-counts the real tracks.
# ============================================================================
# nunique() = number of distinct values in the single column.
n_alone = spots["TRACK_ID"].nunique()

# groupby on the PAIR (image, track) then .ngroups = number of distinct
# (image, track) combinations = the true track count.
n_true = spots.groupby(["SOURCE_IMAGE", "TRACK_ID"]).ngroups

banner("3) Track identity check")
print(f"TRACK_ID alone         : {n_alone}")
print(f"(SOURCE_IMAGE,TRACK_ID) : {n_true}   <- true track count "
      f"({n_true - n_alone} collisions hidden by using TRACK_ID alone)")


# ============================================================================
# 4 & 5. PER-IMAGE SAMPLING + UNLINKED DETECTIONS, MERGED INTO ONE TABLE
# ============================================================================

# --- 4a. One row per image, with frame count and last timestamp -------------
# .agg() lets us compute several summaries at once. Each entry is
#   new_column = (source_column, how_to_aggregate).
per_image = (spots.groupby("SOURCE_IMAGE")
    .agg(
        # FRAME is a 0-based index, so the number of frames is max + 1.
        n_frames=("FRAME", lambda s: int(s.max()) + 1),
        # POSITION_T is absolute time in seconds; the last one is the longest.
        max_T_s =("POSITION_T", "max"),
    ))

# Interval between frames (minutes):
#   total span / number of GAPS / 60.   n_frames-1 gaps, not n_frames — a classic
#   off-by-one trap (5 fence posts have 4 gaps between them).
per_image["interval_min"] = (
    per_image["max_T_s"] / (per_image["n_frames"] - 1) / 60
).round(1)

# --- 4b. Count unlinked detections per image --------------------------------
# isna() finds rows with a missing TRACK_ID (detected, never linked into a track).
# groupby(...).size() then counts those rows per image.
# IMPORTANT: groupby SILENTLY DROPS rows whose key is NaN, which is exactly why
# these detections never reach the track-level analysis — but they DO still count
# as cells in the per-frame counts, so we surface them here.
unlinked = spots[spots["TRACK_ID"].isna()].groupby("SOURCE_IMAGE").size()

# --- 4c. Build a DISPLAY-ONLY copy ------------------------------------------
# We never rename the real per_image index, because later cells look rows up by
# the full SOURCE_IMAGE string (e.g. per_image["n_frames"], intervals[img]) [1].
# Renaming it would quietly break those lookups. So copy first, prettify the copy.
disp = per_image.copy()

# Attach the unlinked counts as a new column.
#   .reindex(disp.index) -> line them up with the SAME image order as disp.
#   .fillna(0)           -> images with zero unlinked spots are MISSING from
#                           `unlinked` (not zero), so fill those gaps with 0.
#   .astype(int)         -> turn 0.0 back into a clean integer.
disp["n_unlinked"] = unlinked.reindex(disp.index).fillna(0).astype(int)

# Flag the odd-cadence images instead of printing all the 30/40 values.
# .mode() = the most common interval (here 40.0); .iloc[0] grabs the first one.
# We convert to string so we can append " *" to any row that differs from it,
# mirroring the "tint the odd one out" idea used in the plots [1].
disp["interval_min"] = disp["interval_min"].astype(str)
usual = disp["interval_min"].mode().iloc[0]
disp.loc[disp["interval_min"] != usual, "interval_min"] += " *"

# Finally swap the 80-char filename index for the short 'MMDD·PositionXXX' label.
disp.index = [short(s) for s in disp.index]
disp.index.name = "image"
assert disp.index.is_unique, "short labels still collide — check the naming rule"
display(
    disp[["n_frames", "interval_min", "n_unlinked"]]
    .style
    .set_caption("4) Per-image sampling  (* = odd cadence)")
    .set_table_styles([{
        "selector": "caption",
        "props": [("caption-side", "top"),
                  ("text-align", "left"),
                  ("font-weight", "bold"),
                  ("font-size", "1.05em"),
                  ("padding-bottom", "6px")],
    }])
)

# --- 5. Headline unlinked figure --------------------------------------------
n_unlinked = int(spots["TRACK_ID"].isna().sum())
print(f"5) Unlinked detections (no TRACK_ID): {n_unlinked} / {len(spots)} "
      f"({100 * n_unlinked / len(spots):.1f}%)  — counted as cells, excluded from motility")

## Cell 3b — Visual QC overview (companion to Cell 3)

Cell 3 prints the integrity checks as tables; this turns the three most useful ones into pictures, because class balance and unevenness are far easier to *see* than to read off a column of numbers. Two figures, four panels.

**Figure 1 — region balance + where the unlinked spots came from.**
- *Left (pie):* the per-spot region counts from Cell 3. A pie is honest here because every spot is in exactly one region (`outside / uninjured / penumbra / core`), so the slices are true parts of one whole (n = total spots). Each wedge is labelled with both its share and its raw count.
- *Right (bar):* the 70 unlinked detections (no `TRACK_ID`) broken down per image, sorted so the worst offenders sit at the top. These are real cells for *counting* but can't contribute to *motility*, so it's worth seeing which fish dominate them before Cell 4 [1].

**Figure 2 — manual curation, uneven vs overall.**
- *Left (100% stacked bar):* per-image manual-vs-automatic, plotted as a **fraction** so a busy fish and a quiet fish compare on the same 0–100 scale — the question is *what proportion was hand-placed*, not *how many spots*. Raw `manual/total` is annotated so nothing is lost, and a dashed line marks the dataset-wide average so each fish reads as above or below it.
- *Right (donut):* the headline split across the whole dataset, with the total in the hole so the denominator is always visible.

The pairing is deliberate: the donut gives the single ~21% figure, but manual placement isn't spread evenly, and the per-image bar is what exposes that unevenness the average hides.

**Why a pie *here* but percentages *there*:** a pie only earns its place when slices are mutually-exclusive parts of one total (region counts, manual-vs-auto). For the per-image curation bar the counts per fish are small, so a percentage with the raw `manual/total` annotated is more honest than a wedge — a "50%" that's really 1-of-2 spots would mislead.

> **Caveat:** the short image label is `MMDD·PositionXXX`, which keeps the two `Position008` fish (July vs October) distinct — the same reused-ID trap Cell 3 flags for `TRACK_ID` [1]. This cell is display-only; everything it draws is recomputed in Cell 3, so it depends on Cell 3 having run.

In [ ]:
import matplotlib.pyplot as plt

# region colours echo the ones the notebook uses elsewhere [1]
REGION_COLORS = {
    "outside tissue":     "lightgrey",
    "uninjured":          "steelblue",
    "injured (penumbra)": "salmon",
    "injury core":        "darkred",
}

# one figure, two panels.  width_ratios gives the bar chart a bit more room
# because nine labels need horizontal space to breathe.
fig, (ax_pie, ax_bar) = plt.subplots(
    1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 1.3]},
    layout="constrained",
)

# ---- LEFT: region class balance (a true part-of-whole -> pie is honest) ----
labels = region_tbl["region"].tolist()
sizes  = region_tbl["n_spots"].tolist()
colors = [REGION_COLORS.get(l, "grey") for l in labels]
total  = sum(sizes)

ax_pie.pie(
    sizes, labels=labels, colors=colors,
    # autopct gets each slice's PERCENT; multiply back up to show the raw count too
    autopct=lambda p: f"{p:.1f}%\n({int(round(p/100*total))})",
    startangle=90, counterclock=False,          # start at top, go clockwise
    wedgeprops={"edgecolor": "white", "linewidth": 1},
    textprops={"fontsize": 9},
)
ax_pie.set_title(f"Region label balance  (n={total} spots)", fontsize=11)
ax_pie.set_anchor("N")    # hang the pie from the top so it lines up with the bars

# ---- RIGHT: where the 70 unlinked detections came from (compare 9 -> bars) --
# sort so the worst offenders are easy to read off the top
unl = disp["n_unlinked"].sort_values()          # ascending -> biggest ends up on top in barh
ax_bar.barh(unl.index, unl.values, color="grey", edgecolor="k", linewidth=0.4)
ax_bar.set_xlabel("unlinked detections (no TRACK_ID)")
ax_bar.set_title(f"Unlinked spots per image  ({int(unl.sum())} total)", fontsize=11)
# annotate each bar with its value so you don't have to eyeball the axis
for y, v in enumerate(unl.values):
    ax_bar.text(v + 0.2, y, str(int(v)), va="center", fontsize=8)
ax_bar.margins(x=0.12)                          # leave room for the labels

fig.tight_layout()
plt.show()




# Same short-label rule as Cell 3, so fish from different dates stay distinct
# (Position008 is reused across imaging sessions) [1].
def short(s):
    parts = s.split("_")
    return f"{parts[0][4:]}·{parts[2]}"        # '0724·Position008'

# ---- per-image manual vs automatic counts --------------------------------
# group the boolean is_manual per image:
#   sum  -> number of True (manual) spots, because booleans sum as 1/0
#   size -> total spots in that image
g = spots.groupby("SOURCE_IMAGE")["is_manual"].agg(manual="sum", total="size")
g["auto"]       = g["total"] - g["manual"]
g["pct_manual"] = 100 * g["manual"] / g["total"]
g.index = [short(s) for s in g.index]          # short labels for the y-axis
g = g.sort_values("pct_manual")                # worst-curated ends up on top in barh

# whole-dataset totals (the headline numbers)
tot_manual = int(g["manual"].sum())
tot_spots  = int(g["total"].sum())
tot_auto   = tot_spots - tot_manual

# one figure, two panels. left is wider: nine labels need horizontal room.
fig, (ax_pie, ax_bar) = plt.subplots(
    1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 1.3]}
)

# ---- LEFT: per-image, as a 100% stacked bar (shows the UNEVENNESS) --------
# We plot PERCENT (not raw counts) so a busy fish and a quiet fish are compared
# on the same 0–100 scale — the question here is "what FRACTION was hand-placed",
# not "how many spots". Raw counts are annotated on the bars so nothing is lost.
y = range(len(g))
pct_manual = g["pct_manual"]
pct_auto   = 100 - pct_manual

ax_bar.barh(y, pct_manual, color="darkorange", edgecolor="k", linewidth=0.4, label="manual")
# left=pct_manual stacks 'auto' starting where 'manual' ends
ax_bar.barh(y, pct_auto, left=pct_manual, color="lightgrey", edgecolor="k",
            linewidth=0.4, label="automatic")

ax_bar.set_yticks(list(y)); ax_bar.set_yticklabels(g.index, fontsize=9)
ax_bar.set_xlabel("% of spots in image"); ax_bar.set_xlim(0, 100)
ax_bar.set_title("Manual curation per image", fontsize=11)
ax_bar.legend(loc="lower right", fontsize=8, framealpha=0.9)

# annotate each bar with the raw "manual/total" so the percentage has context
for yi, (_, row) in zip(y, g.iterrows()):
    ax_bar.text(row["pct_manual"] + 1.5, yi,
                f"{int(row['manual'])}/{int(row['total'])}  ({row['pct_manual']:.0f}%)",
                va="center", fontsize=8)

# dashed line at the dataset-wide manual %, so each fish reads as above/below average
overall_pct = 100 * tot_manual / tot_spots
ax_bar.axvline(overall_pct, ls="--", color="black", lw=1)
ax_bar.text(overall_pct + 1, len(g) - 0.4, f"dataset avg {overall_pct:.1f}%",
            fontsize=8, rotation=90, va="top")

# ---- RIGHT: whole-dataset donut (the headline split) ---------------------
total = tot_spots
ax_pie.pie(
    [tot_manual, tot_auto], labels=["manual", "automatic"],
    colors=["darkorange", "lightgrey"],
    autopct=lambda p: f"{p:.1f}%\n({int(round(p/100*total))})",
    startangle=90, counterclock=False,
    wedgeprops={"width": 0.42, "edgecolor": "white", "linewidth": 1.5},  # width<1 -> donut
    textprops={"fontsize": 9},
)
ax_pie.text(0, 0, f"{total}\nspots", ha="center", va="center", fontsize=11, fontweight="bold")
ax_pie.set_title("Whole dataset", fontsize=11)
ax_pie.set_anchor("N")    # hang the pie from the top so it lines up with the bars

fig.tight_layout()
plt.show()

## Cell 4 — Microglia counts over real time, one panel per image

**The biological question (Q1):** are there more microglia on the injured side, and how does that change over time?

**Counting recipe.** One spot = one cell in one frame, so the number of cells in a (image, frame, region) bucket is just the number of rows in it: `groupby([...]).size()`. To draw a line we need a value at *every* frame including empty ones, so we `pivot` region into columns and `reindex` to the full frame range, filling gaps with 0 (a 0 reads as a point on the line, not a break).

**Two region mappings** are built here from the same per-spot label:
- `region` (simple, for counts): core folds **into** injured → `{0:outside, 1:uninjured, 2:injured, 3:injured}`
- `region_fine` (gradient, for motility later): `{0:outside, 1:uninjured, 2:penumbra, 3:core}`

**Plotting choices**, all deliberate: x = real minutes (frame × that image's interval); identical y-range on every panel so panels are comparable at a glance; a light tint on any image whose cadence differs from the majority, so the odd one out is obvious.

**The new knob — `COUNT_ONLY_TRACKED`.** Set it `True` to count only spots that made it into a track (drops the unlinked detections from Cell 3, item 5); `False` keeps every detection. Default `False` preserves prior behaviour; flip it to see how sensitive the counts are to unlinked spots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ---- knobs ---------------------------------------------------------------
TICK_MIN           = 60
NCOLS              = 2
COUNT_ONLY_TRACKED = True
COLORS = {"injured": "red", "uninjured": "blue", "outside": "lightgrey"}
TINT   = "#fff5e6"
ORDER  = ["injured", "uninjured", "outside"]

REGION      = {0.0: "outside", 1.0: "uninjured", 2.0: "injured",  3.0: "injured"}
REGION_FINE = {0.0: "outside", 1.0: "uninjured", 2.0: "penumbra", 3.0: "core"}
spots["region"]      = spots["MEDIAN_INTENSITY_CH5"].map(REGION)
spots["region_fine"] = spots["MEDIAN_INTENSITY_CH5"].map(REGION_FINE)

count_src = spots[spots["TRACK_ID"].notna()] if COUNT_ONLY_TRACKED else spots

counts = count_src.groupby(["SOURCE_IMAGE", "FRAME", "region"]).size()
YMAX   = int(counts.max()) + 1

intervals     = per_image["interval_min"]
mode_interval = intervals.mode().iloc[0]
XMAX = float((per_image["n_frames"] - 1).mul(intervals).max())

images = sorted(spots["SOURCE_IMAGE"].unique())
nrows  = int(np.ceil(len(images) / NCOLS))
fig, axes = plt.subplots(nrows, NCOLS, figsize=(7 * NCOLS, 2.6 * nrows))
axes = axes.ravel()

plot_rows = []                                              # +++ collect plotted data

for ax, img in zip(axes, images):
    interval = intervals[img]
    sub      = count_src[count_src["SOURCE_IMAGE"] == img]
    frames   = np.arange(int(sub["FRAME"].max()) + 1)
    t_min    = frames * interval

    piv = (sub.groupby(["FRAME", "region"]).size()
              .unstack("region")
              .reindex(index=frames, columns=ORDER)
              .fillna(0))
    for grp in ORDER:
        ax.plot(t_min, piv[grp], marker="o", ms=3, lw=1.5,
                color=COLORS[grp], label=grp)

    # +++ capture EXACTLY what was plotted on this panel (post reindex/fillna)
    long = (piv.reset_index()
               .melt(id_vars="FRAME", var_name="region", value_name="count"))
    long.insert(0, "image", img.split("_")[2])             # short label, as in the title
    long.insert(1, "SOURCE_IMAGE", img)
    long["time_min"]    = long["FRAME"] * interval
    long["interval_min"] = interval
    long["odd_cadence"]  = interval != mode_interval        # the tint flag
    plot_rows.append(long)

    if interval != mode_interval:
        ax.set_facecolor(TINT)
    ax.set_title(f"{img.split('_')[2]}  ({interval:.0f} min)", fontsize=9)
    ax.set_xlabel("Time (min)"); ax.set_ylabel("microglia count")
    ax.set_ylim(0, YMAX); ax.set_xlim(0, XMAX)
    ax.set_xticks(np.arange(0, XMAX + 1, TICK_MIN))
    ax.tick_params(labelsize=8)

for ax in axes[len(images):]:
    ax.axis("off")

handles = [plt.Line2D([], [], color=COLORS[g], marker="o", ms=4, label=g) for g in ORDER]
handles.append(Patch(facecolor=TINT, edgecolor="grey",
                     label=f"odd sampling (\u2260 {mode_interval:.0f} min)"))
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False)
fig.suptitle("Microglia count per region over time, by image", fontsize=12)
fig.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.show()

# +++ assemble the plot-ready table and offer it for download
plot_df = pd.concat(plot_rows, ignore_index=True)
plot_df["count_only_tracked"] = COUNT_ONLY_TRACKED          # record the knob state
plot_df = plot_df[["image", "SOURCE_IMAGE", "FRAME", "time_min",
                   "interval_min", "odd_cadence", "region", "count",
                   "count_only_tracked"]]
export_button(plot_df, "cell4_counts_over_time.csv")

## Cell 5 — Pool the images: mean ± SD count per region over time

Cell 4 shows every fish; this distils them into one curve per region with a spread band. The only hard part is that the fish have different cadences and lengths, so we can't just average "frame 5" across fish. We fix it by **binning onto a common minute grid**:

1. For each fish, assign every frame to a time bin (`(minutes // BIN) * BIN`) and take the **mean count per bin** (so a 30-min and a 40-min fish both contribute one number per bin).
2. *Then* across fish, take **mean, SD and n** per bin.
3. Drop bins where fewer than `MIN_IMAGE_FRAC` of fish contribute — the tail of a timelapse is averaged over very few animals and gets noisy.

New techniques: a two-stage groupby (within-fish, then across-fish), and a `MultiIndex` column frame from `.agg(["mean","std","count"])` — you index it as `summary[(region, "mean")]`.

> **Caveat to keep in mind:** these are **raw counts**, not normalised by region area. If the injured side covers more (or less) tissue than the uninjured side, some of the count difference is just area. Cell 6 repeats the warning; a proper fix is to divide by region area in µm² once you export it.

In [ ]:
BIN_MIN        = 60      # common time-bin width (min)
SHOW_OUTSIDE   = False   # include the 'outside tissue' region?
MIN_IMAGE_FRAC = 0.75    # drop bins where < this fraction of images contribute

mode = "tracked only" if COUNT_ONLY_TRACKED else "all detections"
print(f"counting mode: {mode}  ({len(count_src)} / {len(spots)} spots feed the counts)\n")


# --- 1) per-image, per-bin MEAN count per region --------------------------
per_fish_bin = []
for img in images:
    interval = intervals[img]
    sub      = count_src[count_src["SOURCE_IMAGE"] == img]
    frames   = np.arange(int(sub["FRAME"].max()) + 1)
    piv = (sub.groupby(["FRAME", "region"]).size()
              .unstack("region").reindex(index=frames, columns=ORDER).fillna(0))
    piv["bin"] = ((frames * interval) // BIN_MIN * BIN_MIN).astype(int)
    g = piv.groupby("bin")[ORDER].mean()
    g["SOURCE_IMAGE"] = img
    per_fish_bin.append(g.reset_index())
per_fish_bin = pd.concat(per_fish_bin, ignore_index=True)

# --- 2) across images: mean, SD, n per bin --------------------------------
summary  = per_fish_bin.groupby("bin")[ORDER].agg(["mean", "std", "count"])
n_perbin = summary[(ORDER[0], "count")]
print("images contributing per time bin:")
print(n_perbin.rename("n_images"))

# --- 3) apply the contribution cutoff -------------------------------------
keep    = n_perbin >= MIN_IMAGE_FRAC * len(images)
summary = summary[keep]
print(f"\nkept {keep.sum()}/{len(keep)} bins "
      f"(>= {MIN_IMAGE_FRAC:.0%} of {len(images)} images)")

# --- 4) plot --------------------------------------------------------------
plot_regions = ORDER if SHOW_OUTSIDE else ["injured", "uninjured"]
fig, ax = plt.subplots(figsize=(10, 5))
x = summary.index
for grp in plot_regions:
    m = summary[(grp, "mean")]
    s = summary[(grp, "std")].fillna(0)
    ax.plot(x, m, marker="o", color=COLORS[grp], label=grp)
    ax.fill_between(x, (m - s).clip(lower=0), m + s, color=COLORS[grp], alpha=0.2)
ax.set_xlabel("Time (min)")
ax.set_ylabel(f"mean microglia count  (\u00b1 SD, {BIN_MIN}-min bins)")
ax.set_xticks(np.arange(0, x.max() + 1, TICK_MIN))
ax.set_ylim(bottom=0)
ax.legend(title=f"region (n={len(images)} images)")
ax.set_title("Cross-image mean microglia count per region over time")
fig.tight_layout(); plt.show()


# +++ assemble the plot-ready table and offer it for download
# summary is a MultiIndex-column frame (region × {mean,std,count}); flatten it
# to tidy long form so each row is one region at one time bin = one plotted point.
plot_rows = []
for grp in plot_regions:                         # only the regions actually drawn
    plot_rows.append(pd.DataFrame({
        "bin_min":  summary.index,
        "region":   grp,
        "mean":     summary[(grp, "mean")].values,
        "sd":       summary[(grp, "std")].fillna(0).values,
        "n_images": summary[(grp, "count")].values,
    }))
plot_df = pd.concat(plot_rows, ignore_index=True)

# record the knobs that define this version of the figure
plot_df["bin_min_width"]      = BIN_MIN
plot_df["min_image_frac"]     = MIN_IMAGE_FRAC
plot_df["count_only_tracked"] = COUNT_ONLY_TRACKED
plot_df["show_outside"]       = SHOW_OUTSIDE

export_button(plot_df, "cell5_pooled_counts.csv")

## Cell 6 — Paired per-fish comparison: uninjured vs injured

One summary number per fish per region, analysed as a **paired design** — each fish is its own control, which is far more sensitive than comparing two clouds of points because it cancels the fish-to-fish baseline (some fish are simply busier everywhere).

**Metric:** mean microglia present per frame over the whole timelapse = (total spots in region) / (number of frames). Dividing by frame count makes it independent of how long or how fast each fish was imaged.

**Two panels, two jobs:**
- *Left — slopegraph (the picture).* Each fish's two regions are joined by a line, so you see the *within-animal* change directly. The black line is the group mean. There are **no error bars here on purpose**: two separate per-region CIs would invite the "do they overlap?" eyeball test, which is not a valid test of the difference — so the inference lives entirely in the right panel.
- *Right — paired difference (the inference).* We take the difference `injured − uninjured` **within each fish first**, then build a **single 95% CI** on those *n* differences (mean ± t* × SEM). Subtracting first is what removes the baseline variation; doing two separate CIs would leave it in — that's why the earlier version wasn't really a paired analysis. The dashed line at 0 is the natural null (no within-fish difference): if the interval clears 0, that's the exploratory signal.

This is still **exploratory** — the CI describes the precision of the *mean difference*, it is **not** a hypothesis test. We deliberately don't print a p-value; that comes later with a model that respects the fish-level grouping.

New techniques: `.div(..., axis=0)` to divide each row by a per-row value, within-row subtraction to form the paired differences, and `scipy.stats.t.ppf` for the t multiplier.

> **Caveat (carried from Cell 5):** these are **raw counts**, not normalised by region area. If the injured side covers more (or less) tissue, part of any difference is just area — so a positive interval is suggestive of more microglia on the injured side, not proof. Sign convention: `injured − uninjured`, so positive = more in the injured region.

In [ ]:
from scipy import stats   # only for the t-multiplier of the CI -- still NOT a test

# --- per-fish mean cells/frame in each region (unchanged) -----------------
totals = (count_src[count_src["region"].isin(["uninjured", "injured"])]
          .groupby(["SOURCE_IMAGE", "region"]).size()
          .unstack("region").reindex(columns=["uninjured", "injured"]).fillna(0))
per_fish = totals.div(per_image["n_frames"], axis=0)

# --- THE FIX: one paired difference per fish, then ONE CI on those --------
# Subtracting within each fish removes the fish-to-fish baseline (some fish are
# busy everywhere) — that nuisance variation is exactly what the two separate
# CIs left in, which is why they weren't really a paired analysis [1].
diff   = per_fish["injured"] - per_fish["uninjured"]      # +ve = more in injured region
n      = len(diff)
mean_d = diff.mean()
sem_d  = diff.sem()                                       # std(ddof=1)/sqrt(n)
ci_d   = stats.t.ppf(0.975, df=n - 1) * sem_d
lo, hi = mean_d - ci_d, mean_d + ci_d

fig, (ax_s, ax_d) = plt.subplots(1, 2, figsize=(9, 6),
                                 gridspec_kw={"width_ratios": [1.3, 1]})

# ---- LEFT: slopegraph (the within-fish picture; no per-region CIs) --------
for _, row in per_fish.iterrows():
    ax_s.plot([0, 1], [row["uninjured"], row["injured"]],
              "-", color="grey", alpha=0.5, zorder=1)
ax_s.scatter([0]*n, per_fish["uninjured"], color=COLORS["uninjured"], zorder=2)
ax_s.scatter([1]*n, per_fish["injured"],   color=COLORS["injured"],   zorder=2)
# group means as plain dots — no error bars, so we don't imply the wrong CI
ax_s.plot([0, 1], per_fish[["uninjured", "injured"]].mean().values,
          "-o", color="black", lw=2.5, zorder=3, label="group mean")
ax_s.set_xticks([0, 1]); ax_s.set_xticklabels(["uninjured", "injured"])
ax_s.set_ylabel("mean microglia per frame")
ax_s.set_xlim(-0.4, 1.4); ax_s.set_ylim(bottom=0)
ax_s.set_title("Per-fish, by region"); ax_s.legend()

# ---- RIGHT: the paired difference, with the CI that actually answers it ---
ax_d.axhline(0, ls="--", color="grey", lw=1, zorder=0)    # the null: difference = 0
ax_d.scatter([0]*n, diff, color="steelblue", alpha=0.6, zorder=2)  # one dot per fish
ax_d.errorbar(0, mean_d, yerr=[[mean_d - lo], [hi - mean_d]],
              fmt="o", color="black", capsize=6, lw=2.5, zorder=3,
              label="mean \u00b1 95% CI")
ax_d.set_xticks([]); ax_d.set_xlim(-0.5, 0.5)
ax_d.set_ylabel("injured \u2212 uninjured  (cells/frame)")
ax_d.set_title("Paired difference"); ax_d.legend()
ax_d.legend(loc="upper right")


fig.suptitle(f"Paired comparison, full timelapse  (n = {n} fish)", fontsize=12)
fig.tight_layout(); plt.show()

# --- print the interval so it's not just visual ---------------------------
verdict = "excludes 0" if (lo > 0 or hi < 0) else "includes 0"
print(f"mean paired difference (injured \u2212 uninjured) = {mean_d:.3f} cells/frame")
print(f"95% CI: [{lo:.3f}, {hi:.3f}]  \u2192 {verdict}")

# +++ table 1: the per-fish values behind BOTH panels (slopegraph + difference dots)
per_fish_out = per_fish.reset_index()[["SOURCE_IMAGE", "uninjured", "injured"]].copy()
per_fish_out["image"] = per_fish_out["SOURCE_IMAGE"].str.split("_").str[2]   # short label
per_fish_out["diff_injured_minus_uninjured"] = (
    per_fish_out["injured"] - per_fish_out["uninjured"])
per_fish_out["count_only_tracked"] = COUNT_ONLY_TRACKED
per_fish_out = per_fish_out[["image", "SOURCE_IMAGE", "uninjured", "injured",
                             "diff_injured_minus_uninjured", "count_only_tracked"]]
export_button(per_fish_out, "cell6_per_fish_counts.csv",
              label="⬇  Export per-fish values")

# +++ table 2: the paired-difference summary (the right panel's CI marker)
summary_out = pd.DataFrame([{
    "n_fish":            n,
    "mean_difference":   mean_d,        # injured − uninjured, cells/frame
    "sem":               sem_d,
    "ci95_lo":           lo,
    "ci95_hi":           hi,
    "ci_excludes_zero":  bool(lo > 0 or hi < 0),
    "count_only_tracked": COUNT_ONLY_TRACKED,
}])
export_button(summary_out, "cell6_paired_difference_summary.csv",
              label="⬇  Export CI summary")

## Cell 7 — Build the per-track motility table

This is the workhorse table for the motility half. TrackMate's `tracks` file already has kinematics (speed, confinement, displacement); we **add** the things it doesn't know about and join them on, one row per `(SOURCE_IMAGE, TRACK_ID)`:

1. **Region composition** — what fraction of each track's spots fell in core / penumbra / uninjured / outside (`value_counts().unstack()` → divide by row totals).
2. **A single region label per track**, via a rule you choose: `majority` (where it spent most frames), `end` (where it finished), or `deepest` (the most-injured zone it ever touched). Different biological questions want different rules — that's why it's a knob.
3. **Directionality toward the injury.** TrackMate only gives displacement *magnitude*. But because Step 04 standardises every image to **injury-at-top**, "toward injury" = "moving to smaller Y". So we compute net Y travel from the first and last spot ourselves. `directedness` = (toward-injury distance) / (net displacement) lives in [−1, 1]: +1 = straight at the injury, −1 = straight away, ≈0 = sideways.

> **Why the orientation assumption is safe:** your `04_draw_injury.ijm` flips each stack so injury ends up at the top *before* tracking, and TrackMate runs on the flipped pixels — so the `POSITION_Y` in the CSV is already in the standardised frame. If you ever change that convention, flip `INJURY_AT_TOP`.

New techniques: `.value_counts().unstack(fill_value=0)`, chained `.merge(...)` to assemble a wide table, `np.hypot` for Euclidean distance, and boolean feature columns (`reaches_core`, `is_migrator`).

Quality gate: drop tracks shorter than `MIN_SPOTS` (kinematics from 2–3 points are noise). `MAX_GAPS_ALLOWED` is there if you want gap-free tracks only.

In [ ]:
# ---- knobs ---------------------------------------------------------------
MIN_SPOTS        = 5          # drop tracks shorter than this
MAX_GAPS_ALLOWED = None       # None = keep all; 0 = gap-free only
REGION_RULE      = "majority" # "majority" | "end" | "deepest"
INJURY_AT_TOP    = True       # Step 04 puts injury at top -> toward-injury = decreasing Y

KEYS       = ["SOURCE_IMAGE", "TRACK_ID"]
FINE_ORDER = ["core", "penumbra", "uninjured", "outside"]

# ---- 1) per-track region composition (FINE granularity) ------------------
reg_counts = (spots.groupby(KEYS)["region_fine"]
                   .value_counts().unstack(fill_value=0)
                   .reindex(columns=FINE_ORDER, fill_value=0))
reg_frac = reg_counts.div(reg_counts.sum(axis=1), axis=0)
reg_frac.columns = [f"frac_{c}" for c in reg_frac.columns]

# ---- 1b) one region label per track, per the chosen rule -----------------
if REGION_RULE == "majority":
    region_track = reg_counts.idxmax(axis=1).rename("region_track")
elif REGION_RULE == "end":
    region_track = (spots.sort_values("FRAME").groupby(KEYS).tail(1)
                         .set_index(KEYS)["region_fine"].rename("region_track"))
elif REGION_RULE == "deepest":
    region_track = (spots.groupby(KEYS)["MEDIAN_INTENSITY_CH5"].max()
                         .map(REGION_FINE).rename("region_track"))
else:
    raise ValueError("REGION_RULE must be 'majority', 'end' or 'deepest'")

# ---- 2) fraction of each track's spots that were MANUAL -------------------
frac_manual = spots.groupby(KEYS)["is_manual"].mean().rename("fraction_manual")

# ---- 3) DIRECTIONALITY toward the injury (computed from spots) -----------
ordered = spots.sort_values("FRAME")
first   = ordered.groupby(KEYS)[["POSITION_X", "POSITION_Y"]].first()
last    = ordered.groupby(KEYS)[["POSITION_X", "POSITION_Y"]].last()
dxy = last - first
net_disp = np.hypot(dxy["POSITION_X"], dxy["POSITION_Y"]).rename("net_disp_um")
toward   = (-dxy["POSITION_Y"] if INJURY_AT_TOP else dxy["POSITION_Y"]).rename("toward_injury_um")
directedness = (toward / net_disp).replace([np.inf, -np.inf], np.nan).rename("directedness")
direction = pd.concat([net_disp, toward, directedness], axis=1)

# ---- 4) merge everything onto TrackMate's track features -----------------
per_track = (tracks
             .merge(region_track.reset_index(), on=KEYS)
             .merge(reg_frac.reset_index(),     on=KEYS)
             .merge(frac_manual.reset_index(),  on=KEYS)
             .merge(direction.reset_index(),    on=KEYS))

per_track["frac_injured_side"] = per_track["frac_core"] + per_track["frac_penumbra"]
per_track["reaches_core"]      = per_track["frac_core"] > 0
per_track["is_migrator"]       = ((per_track["frac_uninjured"] > 0) &
                                  (per_track["frac_injured_side"] > 0))

# ---- 5) quality gate -----------------------------------------------------
n_before = len(per_track)
keep = per_track["NUMBER_SPOTS"] >= MIN_SPOTS
if MAX_GAPS_ALLOWED is not None:
    keep &= per_track["NUMBER_GAPS"] <= MAX_GAPS_ALLOWED
per_track = per_track[keep].copy()

# ---- 6) report -----------------------------------------------------------
print(f"tracks: {n_before} -> {len(per_track)} kept "
      f"(MIN_SPOTS={MIN_SPOTS}, MAX_GAPS_ALLOWED={MAX_GAPS_ALLOWED}, REGION_RULE='{REGION_RULE}')")
print("\nregion_track counts (kept):")
print(per_track["region_track"].value_counts())
print(f"\nmigrators (uninjured <-> injured side): {int(per_track['is_migrator'].sum())}")
print(f"tracks reaching the core: {int(per_track['reaches_core'].sum())}")

per_track[KEYS + ["region_track", "frac_uninjured", "frac_penumbra", "frac_core",
                  "is_migrator", "reaches_core", "fraction_manual", "NUMBER_SPOTS",
                  "toward_injury_um", "directedness", "CONFINEMENT_RATIO",
                  "MEAN_STRAIGHT_LINE_SPEED"]].head(12)

## Cell 7b — Core-boundary crossing tracks, per image

A companion to the `is_migrator` flag from Cell 7. That flag fires on *any* uninjured↔injured-side crossing, so a cell that only wobbles between uninjured (1) and penumbra (2) counts as a migrator even though it never touched the lesion. Here the question is narrower and more biological: **how many cells actually cross the core boundary?**

**What counts as a crossing.** We work at the **step level** — a step has two ends, so "crossing" is a property of the *transition*, not of a single frame. For each step we record the region at both ends (`region_start` from the previous frame, `region_end` from this one) and call it a core crossing when **exactly one end is in the core**. The XOR (`^`) is what enforces "exactly one": both-in-core or both-out-of-core → not a crossing; one-in-one-out → a crossing. A penumbra↔uninjured step has *both* ends non-core, so it's never counted — exactly the "not just 1↔2" rule we want.

**Two numbers per fish, because they answer different things:**
- *Bar length* — how many **tracks** cross the core boundary at least once (`.any()` per track). This is the cell count: how many distinct microglia engaged the lesion edge.
- *Annotation `(N steps)`* — the raw number of crossing **events**. A cell that ping-pongs in and out of the core racks up several, so one busy track and several one-time crossers look identical in the bars but mean very different things. Showing both keeps that honest.

Techniques: a self-contained `steps` table carrying region at *both* ends (`region_fine` and its `.shift()`), boolean XOR for the crossing test, and a two-level groupby (`.any()` per track, then summed per image).

> **Caveats.** Gap steps (`dframe > 1`) are *kept* here on purpose — a gapped core→uninjured step still is a crossing topologically, even if we wouldn't trust its speed. This also counts only **kept** tracks (the `MIN_SPOTS` gate from Cell 7), so a fish with many short, discarded tracks will look quieter than it really was. Like everything else here it's a descriptive, per-image view, not a test.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def short(s):
    parts = s.split("_")
    return f"{parts[0][4:]}·{parts[2]}"        # distinct across dates [1]

KEYS = ["SOURCE_IMAGE", "TRACK_ID"]

# self-contained per-step table carrying region at BOTH ends of each step
s   = spots.merge(per_track[KEYS].drop_duplicates(), on=KEYS, how="inner")
s   = s.sort_values(KEYS + ["FRAME"])
grp = s.groupby(KEYS, sort=False)
s["region_end"]   = s["region_fine"]             # where the step LANDS (this frame)
s["region_start"] = grp["region_fine"].shift()   # where it BEGAN (previous frame) [1]
steps = s.dropna(subset=["region_start"])

# a step crosses the CORE boundary when exactly one end is in the core.
# XOR (^) is the key: both-in or both-out -> False; one-in-one-out -> True.
# A penumbra<->uninjured step has both ends non-core -> never counted, which is
# exactly the "not just 1<->2" rule you asked for.
in_start = steps["region_start"] == "core"
in_end   = steps["region_end"]   == "core"
steps = steps.assign(core_cross=in_start ^ in_end)

# --- per image: how many tracks cross the core boundary at least once ------
crossed_track = steps.groupby(KEYS)["core_cross"].any()   # one bool per track
crossers = crossed_track.groupby(level=0).sum()           # per image (level 0 = SOURCE_IMAGE)
totals   = crossed_track.groupby(level=0).size()
# and the raw number of crossing EVENTS (steps), which counts ping-ponging
cross_events = steps.groupby("SOURCE_IMAGE")["core_cross"].sum()

m = totals.to_frame("total_tracks")
m["crossers"]    = crossers
m["non_cross"]   = m["total_tracks"] - m["crossers"]
m["cross_steps"] = cross_events.reindex(m.index).fillna(0).astype(int)
m.index = [short(i) for i in m.index]
m = m.sort_values("crossers")

fig, ax = plt.subplots(figsize=(8, 5))
y = range(len(m))
ax.barh(y, m["crossers"], color="darkred", edgecolor="k", linewidth=0.4, label="crosses core boundary")
ax.barh(y, m["non_cross"], left=m["crossers"], color="lightgrey", edgecolor="k",
        linewidth=0.4, label="never enters/leaves core")
ax.set_yticks(list(y)); ax.set_yticklabels(m.index, fontsize=9)
ax.set_xlabel("tracks (kept)")
ax.set_title(f"Core-boundary crossing tracks per image  "
             f"({int(m['crossers'].sum())}/{int(m['total_tracks'].sum())} tracks)")
ax.legend(loc="lower right", fontsize=8)
# annotate "crossing tracks / total  (N crossing steps)" so ping-pongers show up
for yi, (_, r) in zip(y, m.iterrows()):
    ax.text(r["total_tracks"] + 0.05, yi,
            f"{int(r['crossers'])}/{int(r['total_tracks'])}  ({int(r['cross_steps'])} steps)",
            va="center", fontsize=8)
ax.margins(x=0.18)
fig.tight_layout(); plt.show()

## Cell 12 — Per-image trajectory maps

A direct, qualitative look that complements every aggregate above: **draw each cell's path**, one panel per fish. This is where the abstractions (speed, MSD α, region labels) become a picture you can actually point at.

**Three things are encoded at once, each on a different element:**
- **Line colour = cell identity.** Every track in a panel gets its own colour (from `tab20`), so when paths overlap you can still trace one cell's whole journey by following its thread — the previous single grey line made overlapping cells indistinguishable.
- **Soft ribbon = territory covered.** The path is drawn a second time underneath, much thicker and translucent, with round caps/joins so it reads as a smooth "sausage" hugging exactly where the cell went. A confined cell becomes a small tight smudge; a directed one a long streak — the same confined-vs-directed contrast as the MSD plot, but visible per cell. Where two ribbons overlap, the colours mix, showing shared ground.
- **Dot colour = region per frame.** Each dot is one frame, coloured by the region the cell was in at that moment, so you can watch a track change colour as it crosses toward the injury.

**Reading tips:**
- The Y-axis runs **high-to-low** (`set_ylim(CANVAS_Y_UM, 0)`) so small Y / the injury sits at the **top**, matching the microscope image — a track drifting *upward* is heading *toward* the injury.
- Panels are pinned to the **true image canvas** (1024×512 px → 290.91×145.45 µm, supplied from FIJI since the data can't carry it). With `set_aspect("equal")` this means every fish is drawn at the same, undistorted scale, so positions are comparable *between* panels, not just shapes within one.

> **Caveats.** The ribbon is a *qualitative* coverage cue: its width is in points (a fixed visual size), **not** measured microns, so don't read its area as a real µm² territory — the honest single-number cousin is the radius of gyration, the quantity that bridges back to the MSD plateau. `tab20` repeats after 20 cells, so a panel with >20 kept tracks could reuse a hue. The territory follows the path itself, so unlike a convex hull it never shades ground the cell didn't cross.

In [ ]:
from matplotlib.lines import Line2D

# ---- knobs ---------------------------------------------------------------
NCOLS_TRAJ     = 3
MIN_SPOTS_TRAJ = 3        # skip 1-2 point fragments (no real path)
PATH_LW        = 1.3      # the crisp trajectory line
RIBBON_LW      = 13        # the soft "territory" ribbon underneath (points)
RIBBON_ALPHA   = 0.18     # how faint the ribbon is
DOT_SIZE       = 12
FINE_COLORS = {"core": "darkred", "penumbra": "red", "uninjured": "blue", "outside": "lightgrey"}

# Full image canvas in microns (FIJI: 1024x512 px -> 290.91 x 145.45 um)
CANVAS_X_UM = 290.91
CANVAS_Y_UM = 145.45

TRACK_CMAP = plt.get_cmap("tab10")    # 10 distinct hues, no dark/light pairs

traj = spots[spots["TRACK_ID"].notna()].copy().sort_values(KEYS + ["FRAME"])
imgs = sorted(traj["SOURCE_IMAGE"].unique())
nrows = int(np.ceil(len(imgs) / NCOLS_TRAJ))
fig, axes = plt.subplots(nrows, NCOLS_TRAJ, figsize=(5 * NCOLS_TRAJ, 4.3 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, img in zip(axes, imgs):
    sub  = traj[traj["SOURCE_IMAGE"] == img]
    tids = [tid for tid, d in sub.groupby("TRACK_ID") if len(d) >= MIN_SPOTS_TRAJ]
    assert len(tids) <= TRACK_CMAP.N, f"{img}: {len(tids)} tracks > {TRACK_CMAP.N} colours — hues will repeat"
    for k, tid in enumerate(tids):
        d = sub[sub["TRACK_ID"] == tid]
        x = d["POSITION_X"].to_numpy(); y = d["POSITION_Y"].to_numpy()
        tcolor = TRACK_CMAP(k % TRACK_CMAP.N)               # this cell's identity colour

        # soft fat ribbon = territory echo; round caps/joins so it's a smooth sausage
        ax.plot(x, y, "-", color=tcolor, lw=RIBBON_LW, alpha=RIBBON_ALPHA,
                solid_capstyle="round", solid_joinstyle="round", zorder=0)
        # crisp thin path on top, same colour = cell identity
        ax.plot(x, y, "-", color=tcolor, lw=PATH_LW, alpha=0.9, zorder=1)
        # per-frame dots = region
        ax.scatter(x, y, c=d["region_fine"].map(FINE_COLORS).to_numpy(),
                   s=DOT_SIZE, zorder=2, edgecolor="k", linewidth=0.2)

    ax.set_title(img.split("_")[2], fontsize=9)
    ax.set_aspect("equal")
    ax.set_xlim(0, CANVAS_X_UM)
    ax.set_ylim(CANVAS_Y_UM, 0)        # inverted: injury (small Y) at top [1]
    ax.set_xlabel("X (\u00b5m)"); ax.set_ylabel("Y (\u00b5m)"); ax.tick_params(labelsize=7)

for ax in axes[len(imgs):]:
    ax.axis("off")

handles = [Line2D([], [], marker="o", ls="", color=FINE_COLORS[r], mec="k", label=r)
           for r in ["core", "penumbra", "uninjured", "outside"]]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=9)
fig.suptitle("Microglia trajectories per image  (ribbon = path territory; line = cell, dot = region)",
             fontsize=13)
fig.tight_layout(rect=[0, 0.04, 1, 0.97]); plt.show()

## Cell 9b — Per-track motility, two groupings side by side

A direct extension of Cell 9. The left panel is the familiar three-group gradient (uninjured → penumbra → core); the right panel collapses it to the blunt **core vs rest** contrast, both drawn from the *same* tracks so you can see at a glance whether merging uninjured + penumbra throws any structure away.

**What each dot is.** One jittered dot per **track**, the black bar is the **median** and the thin line the **inter-quartile range** (median + IQR, not mean + SD, because track metrics are skewed). The two panels share a y-axis, so "higher = faster" reads identically across both and the medians are directly comparable.

**The grouping.** The binary map is the same split as Cell 13 with `PENUMBRA_AS_INJURY = False`: core (region 3) is "injury core"; uninjured + penumbra (1 + 2) become "outside core" [1]. Reading the two panels together is the point — if the penumbra looks like a twin of uninjured, the collapse is faithful; if it sits apart (e.g. as the fastest group), the binary view is hiding a real intermediate state, and the `n=` labels make explicit that "outside core" merges two of the three columns.

Swap `METRIC` to ask different questions (`CONFINEMENT_RATIO`, `directedness`, `toward_injury_um`, …); `SCALE = 3600` converts speed columns µm/s → µm/hour, `1.0` otherwise.

> **Two caveats, both carried from earlier.** This is the **cell-level** view: each track wears its **majority** region label (Cell 7's default rule), so a boundary-crossing cell is filed wholly under whichever region it spent most frames in — the contamination the step-level companion removes [1]. And pooling treats every track as independent, ignoring that tracks are nested within fish; fine for exploration, but a final statistic needs a mixed-effects model with fish as a random effect [1].

In [ ]:
METRIC     = "TRACK_MEDIAN_SPEED"   # try: CONFINEMENT_RATIO, directedness, toward_injury_um
SCALE      = 3600.0                 # um/s -> um/hour (use 1.0 for non-speed metrics)
YLABEL     = "median step speed (\u00b5m/hour)"
MOT_COLORS = {"core": "darkred", "penumbra": "red", "uninjured": "blue"}

# --- three-group data (unchanged) ----------------------------------------
REGIONS = ["uninjured", "penumbra", "core"]
pdf = per_track[per_track["region_track"].isin(REGIONS)].copy()
pdf["_y"] = pdf[METRIC] * SCALE

# --- binary regrouping: core (code 3) vs uninjured+penumbra (codes 1,2) ---
BINARY = {"core": "injury core", "penumbra": "outside core", "uninjured": "outside core"}
pdf["region_binary"] = pdf["region_track"].map(BINARY)
BIN_ORDER  = ["outside core", "injury core"]              # left -> right
BIN_COLORS = {"outside core": "steelblue", "injury core": "darkred"}

# entered the core at some point, but majority region is NOT core [1]
pdf["entered_not_major"] = pdf["reaches_core"] & (pdf["region_track"] != "core")

rng = np.random.default_rng(0)       # reproducible jitter

def panel(ax, group_col, order, colors, title, highlight=None):
    """dot=track, black bar=median, thin line=IQR; highlight=col -> red outline."""
    for i, grp in enumerate(order):
        sub = pdf[pdf[group_col] == grp].dropna(subset=["_y"])
        if sub.empty:
            continue
        g = sub["_y"].to_numpy()
        x = i + rng.uniform(-0.12, 0.12, len(g))
        if highlight is not None:                      # per-dot edge styling
            hl = sub[highlight].to_numpy()
            ec = np.where(hl, "red", "black")
            lw = np.where(hl, 1.6, 0.3)
        else:
            ec, lw = "black", 0.3
        ax.scatter(x, g, color=colors[grp], alpha=0.7,
                   edgecolor=ec, linewidth=lw, zorder=2)
        med = np.median(g); q1, q3 = np.percentile(g, [25, 75])
        ax.plot([i-0.22, i+0.22], [med, med], color="k", lw=2.5, zorder=3)
        ax.plot([i, i], [q1, q3], color="k", lw=1.2, zorder=3)
        ax.text(i+0.28, med, f"n={len(g)}", va="center", fontsize=8)
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order)
    ax.set_xlim(-0.5, len(order)-0.5)
    ax.set_title(title, fontsize=11)

fig, (ax3, ax2) = plt.subplots(1, 2, figsize=(11, 5), sharey=True,
                               gridspec_kw={"width_ratios": [1.4, 1]})

panel(ax3, "region_track",  REGIONS,   MOT_COLORS, "Three-group gradient")
panel(ax2, "region_binary", BIN_ORDER, BIN_COLORS, "Binary: core vs rest",
      highlight="entered_not_major")

# proxy legend handle for the red outline
from matplotlib.lines import Line2D
ax2.legend(handles=[Line2D([], [], marker="o", ls="", mfc="lightgrey",
                           mec="red", mew=1.6,
                           label="entered core (not majority)")],
           loc="upper right", fontsize=8)

ax3.set_ylabel(YLABEL)                 # shared y -> label only on the left
fig.suptitle(f"Pooled per-track motility: {METRIC}  (dot = track, n={len(pdf)})", fontsize=12)
fig.tight_layout(); plt.show()

# +++ per-track values behind BOTH panels (each row = one dot)
plot_df = pdf[["SOURCE_IMAGE", "TRACK_ID", "region_track", "region_binary",
               "reaches_core", "entered_not_major", "_y"]].copy()
plot_df["image"] = plot_df["SOURCE_IMAGE"].str.split("_").str[2]   # short label
plot_df = plot_df.rename(columns={"_y": "value"})
plot_df["metric"] = METRIC                  # which column _y came from
plot_df["scale"]  = SCALE                   # the unit conversion applied
plot_df = plot_df[["image", "SOURCE_IMAGE", "TRACK_ID",
                   "region_track", "region_binary", "reaches_core",
                   "entered_not_major", "metric", "scale", "value"]]
export_button(plot_df, "cell9b_per_track_motility.csv",
              label="⬇  Export per-track values")

# +++ the per-group summary stats drawn on each panel (median + IQR + n)
sum_rows = []
for col, order in [("region_track", REGIONS), ("region_binary", BIN_ORDER)]:
    for grp in order:
        g = pdf.loc[pdf[col] == grp, "_y"].dropna()
        if len(g):
            q1, q3 = g.quantile([0.25, 0.75])
            sum_rows.append({"grouping": col, "group": grp, "n": len(g),
                             "median": g.median(), "q1": q1, "q3": q3})
summary_df = pd.DataFrame(sum_rows)
summary_df["metric"] = METRIC
export_button(summary_df, "cell9b_motility_summary.csv",
              label="⬇  Export median/IQR summary")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# ---- knobs ---------------------------------------------------------------
DWELL_REGION = "core"     # the injury ROI = region_fine label 3 (Step 04)
KEYS = ["SOURCE_IMAGE", "TRACK_ID"]

# per-image cadence (min/frame) — dwell must be in real minutes, not frames
_p = spots.groupby("SOURCE_IMAGE").agg(_n=("FRAME", lambda s: int(s.max())+1),
                                       _t=("POSITION_T", "max"))
interval_min = (_p["_t"] / (_p["_n"] - 1) / 60).round().astype(int)

# tracked spots only; region is already per-frame in region_fine
tk = (spots.merge(per_track[KEYS].drop_duplicates(), on=KEYS, how="inner")
           .sort_values(KEYS + ["FRAME"]))

rows = []
for keys, d in tk.groupby(KEYS, sort=False):
    in_core = (d["region_fine"] == DWELL_REGION).to_numpy()
    n_core  = int(in_core.sum())
    if n_core == 0:
        continue                                 # never entered the ROI
    iv = interval_min[keys[0]]
    rows.append({"SOURCE_IMAGE": keys[0], "TRACK_ID": keys[1],
                 "image": keys[0].split("_")[2], "interval_min": iv,
                 "n_core_frames": n_core, "dwell_min": n_core * iv})

dwell = pd.DataFrame(rows)

# tag each entrant: is the core its MAJORITY region (resident) or not (visitor)?
maj = per_track[KEYS + ["region_track"]].copy()
dwell = dwell.merge(maj, on=KEYS, how="left")
dwell["resident"] = dwell["region_track"] == DWELL_REGION   # majority-core = resident

# ---- strip plot ----------------------------------------------------------
res  = dwell["resident"].to_numpy()
vals = dwell["dwell_min"].to_numpy()
rng  = np.random.default_rng(0)
x    = rng.uniform(-0.12, 0.12, len(dwell))

fig, ax = plt.subplots(figsize=(4.5, 6))
# solid = resident (majority core); hollow = visitor (entered, lives elsewhere)
ax.scatter(x[res], vals[res], color="darkred", alpha=0.8, edgecolor="k",
           linewidth=0.3, zorder=2, label=f"resident (majority core, n={res.sum()})")
ax.scatter(x[~res], vals[~res], facecolor="none", edgecolor="darkred",
           linewidth=1.2, zorder=2, label=f"visitor (entered only, n={(~res).sum()})")

med = np.median(vals); q1, q3 = np.percentile(vals, [25, 75])
ax.plot([-0.22, 0.22], [med, med], color="k", lw=2.5, zorder=3)   # median (all entrants)
ax.plot([0, 0], [q1, q3], color="k", lw=1.2, zorder=3)            # IQR
ax.text(0.28, med, f"median {med:.0f} min\n(all, n={len(dwell)})", va="center", fontsize=9)

ax.set_xticks([0]); ax.set_xticklabels(["injury core"])
ax.set_xlim(-0.5, 0.85); ax.set_ylim(bottom=0)
ax.set_ylabel("time in injury core (min)")
ax.yaxis.set_major_locator(MultipleLocator(60))      # a tick every hour (60 min)
ax.set_title(f"Dwell time per cell that entered the core  (n={len(dwell)})")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout(); plt.show()

# +++ export
export_button(dwell.assign(dwell_region=DWELL_REGION),
              "cell9c_core_dwell.csv", label="⬇  Export dwell times")

## Cell 10b — Step-speed by region, two groupings side by side

A fold-together of the Cell 10 ridge and its binary collapse. The left panel is the full three-group gradient (uninjured / penumbra / core); the right collapses it to **core vs rest**, both drawn from the *same* per-step speeds so you can read directly whether merging uninjured + penumbra throws any structure away.

**What a step is (carried from Cell 10).** Each step is one frame-to-frame jump: distance `np.hypot(dx, dy)` over elapsed time `dt`, attributed to the region at its **start** (`region_fine.shift()`) [1]. Gap steps span 2+ frames and so underestimate speed; with `EXCLUDE_GAP_STEPS = True` we keep only clean 1-frame steps, matching the rule that gapped steps are kept for QC but excluded from motility [1].

**One knob block, two panels.** `PENUMBRA_AS_INJURY` and the smoothing/overlap settings drive both ridges, and the binary relabel is a single `.map` on the same `steps` table — so the only thing differing between the panels is the *grouping*, never a recompute. The two panels share one x-axis (a shared `vmax` across all steps), which is what makes them honestly comparable: the binary core ridge sits at exactly the same speed scale as the three-group one. The dashed tick on each ridge is that group's median.

**Reading it against the hypothesis "side doesn't matter, only in-vs-out of core."** If uninjured and penumbra are near-twins (overlapping shapes, close medians), the collapse loses nothing and the right panel faithfully compresses the left. If the penumbra sits *apart* — e.g. as the fastest group, with its median shifted right of uninjured — then the side *does* matter, and the binary view is averaging away a real intermediate state.

> **Caveats.** This pools *all* steps and treats them as independent, ignoring that steps are nested within tracks within fish — so it's the right exploratory picture, not a test; a proper contrast needs a mixed-effects model with fish as a random effect. The KDE shape is also bandwidth-dependent (`SMOOTHING`): too small invents bumps, too large erases real ones — worth nudging it and checking the medians (printed below) don't move.

In [ ]:
from scipy.stats import gaussian_kde
import numpy as np
import matplotlib.pyplot as plt

# ============================================================================
# ONE KNOB BLOCK — drives BOTH panels (left = 3-group gradient, right = binary)
# ============================================================================
KEYS = ["SOURCE_IMAGE", "TRACK_ID"]
EXCLUDE_GAP_STEPS  = True    # 1-frame steps only (recommended for motility) [1]
SMOOTHING          = 0.15     # KDE bandwidth: smaller = more detail
OVERLAP            = 1.4     # ridge height vs row spacing
PENUMBRA_AS_INJURY = False   # False: inside = core only; True: inside = penumbra+core [1]

# ---- step-level region maps, both derived from the SAME switch ------------
REGIONS    = ["uninjured", "penumbra", "core"]                 # left panel, top->bottom
MOT_COLORS = {"core": "darkred", "penumbra": "red", "uninjured": "blue"}

INSIDE  = ["core", "penumbra"] if PENUMBRA_AS_INJURY else ["core"]
OUTSIDE = [r for r in ["uninjured", "penumbra"] if r not in INSIDE]
IN_LABEL  = "injured side" if PENUMBRA_AS_INJURY else "injury core"
OUT_LABEL = "uninjured"    if PENUMBRA_AS_INJURY else "outside core"
ORDER2     = [OUT_LABEL, IN_LABEL]                             # right panel, top->bottom
BIN_COLORS = {IN_LABEL: "darkred", OUT_LABEL: "steelblue"}

def to_binary(fine):
    if fine in INSIDE:  return IN_LABEL
    if fine in OUTSIDE: return OUT_LABEL
    return None                                                # region 0 dropped [1]

# ---- per-step speeds, computed ONCE, fed to both panels ------------------
# region attributed to where the step STARTED (region_fine.shift), the Cell 10 rule [1]
s   = spots.merge(per_track[KEYS].drop_duplicates(), on=KEYS, how="inner").sort_values(KEYS + ["FRAME"])
grp = s.groupby(KEYS, sort=False)
s["dist_um"]      = np.hypot(grp["POSITION_X"].diff(), grp["POSITION_Y"].diff())
s["dt_s"]         = grp["POSITION_T"].diff()
s["dframe"]       = grp["FRAME"].diff()
s["region_start"] = grp["region_fine"].shift()
steps = s.dropna(subset=["dist_um", "dt_s", "region_start"])
steps = steps[steps["dt_s"] > 0]
if EXCLUDE_GAP_STEPS:
    steps = steps[steps["dframe"] == 1]
steps = steps.assign(speed_um_hour=steps["dist_um"] / steps["dt_s"] * 3600.0,
                     bin_region=lambda d: d["region_start"].map(to_binary))

# ---- ONE x-axis for both panels, so the ridges are directly comparable ----
# Using a SHARED vmax (across all steps) is what makes the two panels honest:
# the binary core ridge sits at exactly the same speed scale as the 3-group one.
vmax = np.nanpercentile(steps["speed_um_hour"], 99)
xs   = np.linspace(0, vmax, 300)

def draw_ridge(ax, data, col, order, colors, title):
    """One ridge panel. data=steps subset, col=grouping column, order=top->bottom."""
    n = len(order)
    dens, cnt = {}, {}
    for r in order:
        v = data.loc[data[col] == r, "speed_um_hour"].dropna().values
        cnt[r]  = len(v)
        dens[r] = (gaussian_kde(v, bw_method=SMOOTHING)(xs)
                   if (len(v) >= 2 and np.std(v) > 0) else np.zeros_like(xs))
    gmax = max((d.max() for d in dens.values()), default=1.0) or 1.0
    for idx, r in enumerate(order):
        base = n - 1 - idx
        d    = dens[r] / gmax * OVERLAP
        ax.fill_between(xs, base, base + d, color=colors[r], alpha=0.8, zorder=idx + 2)
        ax.plot(xs, base + d, color="k", lw=0.8, zorder=idx + 2)
        v = data.loc[data[col] == r, "speed_um_hour"].dropna()
        if len(v):                                            # dashed median tick
            ax.plot([v.median()] * 2, [base, base + 0.18 * OVERLAP],
                    color="k", lw=1.4, ls="--", zorder=idx + 2.1)
    ax.set_yticks([n - 1 - i for i in range(n)])
    ax.set_yticklabels([f"{r}\n(steps={cnt[r]})" for r in order])
    for tick, r in zip(ax.get_yticklabels(), order):
        tick.set_color(colors[r]); tick.set_fontsize(9)
    ax.set_ylim(-0.2, n - 1 + OVERLAP + 0.3)
    ax.set_xlabel("step speed (\u00b5m/hour)")
    ax.set_title(title, fontsize=11)

# right panel is shorter (2 ridges) but shares the x-axis with the left
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
draw_ridge(axL, steps, "region_start", REGIONS, MOT_COLORS,
           "Three-group gradient\n(does the side matter?)")
draw_ridge(axR, steps.dropna(subset=["bin_region"]), "bin_region", ORDER2, BIN_COLORS,
           f"Binary collapse\n({IN_LABEL} vs {OUT_LABEL})")
axR.set_xlim(0, vmax)
fig.suptitle(f"Step-speed by region, two groupings  "
             f"(smoothing={SMOOTHING}, gap steps excluded={EXCLUDE_GAP_STEPS})", fontsize=12)
fig.tight_layout(); plt.show()

# numbers under the figure, for both groupings
print("three-group:")
print(steps.groupby("region_start")["speed_um_hour"].agg(["count", "median", "mean"]).round(1))
print("\nbinary:")
print(steps.dropna(subset=["bin_region"]).groupby("bin_region")["speed_um_hour"]
      .agg(["count", "median", "mean"]).round(1))


# +++ table 1: the per-step speeds behind BOTH ridge panels (each row = one step)
# Both panels are KDEs of these same speeds; they differ only in the grouping
# column, so carrying both labels per step reconstructs either panel.
steps_out = steps[["SOURCE_IMAGE", "TRACK_ID", "FRAME",
                   "region_start", "bin_region", "dframe",
                   "dist_um", "dt_s", "speed_um_hour"]].copy()
steps_out["image"] = steps_out["SOURCE_IMAGE"].str.split("_").str[2]   # short label
steps_out["exclude_gap_steps"]   = EXCLUDE_GAP_STEPS
steps_out["penumbra_as_injury"]  = PENUMBRA_AS_INJURY
export_button(steps_out, "cell10b_step_speeds.csv",
              label="⬇  Export per-step speeds")

# +++ table 2: the summary medians/means drawn under the figure (both groupings)
three = (steps.groupby("region_start")["speed_um_hour"]
              .agg(["count", "median", "mean"]).reset_index()
              .rename(columns={"region_start": "group"}))
three["grouping"] = "three_group"
binr  = (steps.dropna(subset=["bin_region"]).groupby("bin_region")["speed_um_hour"]
              .agg(["count", "median", "mean"]).reset_index()
              .rename(columns={"bin_region": "group"}))
binr["grouping"] = "binary"
summary_out = pd.concat([three, binr], ignore_index=True)
summary_out["penumbra_as_injury"] = PENUMBRA_AS_INJURY
export_button(summary_out, "cell10b_step_speed_summary.csv",
              label="⬇  Export median/mean summary")


## Cell 11b — Mean Squared Displacement, two groupings side by side

A fold-together of the per-region MSD curve and its binary collapse. **MSD** answers a question raw speed can't: *after a time lag τ, how far has a cell typically moved, squared?* The shape of MSD vs τ tells you the **style** of motion, not just its pace:

| MSD vs τ shape | log-log slope α | Motion type |
|----------------|-----------------|-------------|
| bends down, flattens to a plateau | α < 1 | confined — trapped in a region |
| straight line | α ≈ 1 | diffusive — a random walk |
| bends up | α > 1 (→2) | directed — persistent travel toward a target |

So MSD separates a cell that *jiggles fast but goes nowhere* (confined, low α) from one that *creeps steadily toward the injury* (directed, high α).

**How it's computed (carried from the original MSD cell).** Each track's MSD is **time-averaged** — every pair of points τ frames apart contributes — then **ensemble-averaged** within a region. It's **gap-aware**: points are indexed by frame number, so a cell missing in one frame never has its neighbours treated as adjacent (which would corrupt every lag). Lag is converted frames → **minutes** per image and binned onto a common grid so the 30- and 40-min fish pool [1]. α is fit over **short lags only**, where many tracks contribute; long lags are too sparse to trust.

**One knob block, two panels.** The MSD curves are built **once**; the left panel groups them by the full `uninjured / penumbra / core` gradient, the right by the binary `PENUMBRA_AS_INJURY` split — the binary relabel is a single `.map` on the same table, so the only difference between panels is the grouping. Both share **log x and log y axes** (`sharex`/`sharey`), which matters more here than elsewhere: because the *slope* is the message, a shared axis lets you lay a mental straight-edge across both and trust that a steeper curve really is more directed. Faded error bars are SEM across tracks; the dashed line is the fitted power law.

**Reading it against the hypothesis.** If uninjured and penumbra curl up together (similar steep slopes) and core is the shallow odd-one-out, then for *directedness* the binary collapse is faithful — the "outside core" α is essentially the average of the two it merges. Note this can be the *opposite* of the step-speed story, where the penumbra was the fastest group and the collapse lost that peak: the penumbra can resemble uninjured in **directedness** while differing in **raw speed**.

> **Caveats.** Each track wears its **majority** region label, so a boundary-crosser's whole MSD curve (including displacement that happened elsewhere) lands in one bucket — MSD needs the full trajectory, so majority is the only practical track-level option. Read α as a **direction of evidence**, not a verdict — with this few tracks per region the curve *shape* is often more honest than the fitted number, and pooling ignores fish-level nesting, so this is exploratory rather than a test [1].

In [ ]:
from scipy.stats import gaussian_kde   # not used here, but harmless if already imported
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================================
# ONE KNOB BLOCK — drives BOTH panels (left = 3-group, right = binary)
# ============================================================================
KEYS = ["SOURCE_IMAGE", "TRACK_ID"]
MAX_LAG_FRAC       = 0.5     # use lags up to 50% of each track's span
MIN_SPOTS_MSD      = 5       # ignore very short tracks
LAG_BIN_MIN        = 20      # round lag-time to this grid so 30- & 40-min fish pool
FIT_MAX_LAG_MIN    = 160     # fit alpha (log-log slope) over short lags only
MIN_TRACKS_PER_LAG = 3       # only plot a lag if >= this many tracks reach it
PENUMBRA_AS_INJURY = False   # False: inside = core only; True: inside = penumbra+core [1]

REGIONS_MSD = ["uninjured", "penumbra", "core"]
MOT_COLORS  = {"core": "darkred", "penumbra": "red", "uninjured": "blue", "outside": "grey"}

INSIDE  = ["core", "penumbra"] if PENUMBRA_AS_INJURY else ["core"]
OUTSIDE = [r for r in ["uninjured", "penumbra"] if r not in INSIDE]
IN_LABEL  = "injured side" if PENUMBRA_AS_INJURY else "injury core"
OUT_LABEL = "uninjured"    if PENUMBRA_AS_INJURY else "outside core"
ORDER2     = [OUT_LABEL, IN_LABEL]
BIN_COLORS = {IN_LABEL: "darkred", OUT_LABEL: "steelblue"}

def to_binary(fine):
    if fine in INSIDE:  return IN_LABEL
    if fine in OUTSIDE: return OUT_LABEL
    return None                                       # region 0 -> dropped [1]

# ---- per-image real-time interval (min); cadence differs across images ----
_p = spots.groupby("SOURCE_IMAGE").agg(_n=("FRAME", lambda s: int(s.max())+1),
                                       _t=("POSITION_T", "max"))
interval_min = (_p["_t"] / (_p["_n"] - 1) / 60).round().astype(int)

# ---- one region label per track (majority of the fine label) [1] ----------
tracked = spots[spots["TRACK_ID"].notna()].copy()
region_of_track = tracked.groupby(KEYS)["region_fine"].agg(lambda s: s.value_counts().idxmax())

def time_averaged_msd(d, max_lag_frac=MAX_LAG_FRAC):
    # Gap-aware time-averaged MSD for ONE track: index by FRAME so missing
    # frames are never treated as adjacent; average sq displacement over all
    # real pairs exactly tau apart [1].
    d = d.dropna(subset=["POSITION_X", "POSITION_Y"])
    f = d["FRAME"].astype(int).to_numpy()
    pos = {int(fr): (x, y) for fr, x, y in zip(f, d["POSITION_X"], d["POSITION_Y"])}
    fmin, fmax = f.min(), f.max(); span = fmax - fmin
    out = {}
    for lag in range(1, int(span * max_lag_frac) + 1):
        sq = [(pos[fr+lag][0]-pos[fr][0])**2 + (pos[fr+lag][1]-pos[fr][1])**2
              for fr in range(fmin, fmax - lag + 1) if fr in pos and fr+lag in pos]
        if sq:
            out[lag] = np.mean(sq)
    return out

# ---- build the per-track MSD curves ONCE; both panels read from this -------
rows = []
for keys, d in tracked.groupby(KEYS):
    if len(d) < MIN_SPOTS_MSD:
        continue
    iv = interval_min[keys[0]]
    for lag, m in time_averaged_msd(d).items():
        rows.append((keys[0], keys[1], region_of_track.loc[keys], lag, lag*iv, m))
msd = pd.DataFrame(rows, columns=["SOURCE_IMAGE", "TRACK_ID", "region",
                                  "lag_frame", "lag_min", "msd_um2"])
msd["lag_bin"]    = (np.round(msd["lag_min"] / LAG_BIN_MIN) * LAG_BIN_MIN).astype(int)
msd["bin_region"] = msd["region"].map(to_binary)     # the binary relabel, added once

# ---- helper: draw one MSD panel, fit alpha, return the alpha dict ----------
def draw_msd(ax, data, col, order, colors, title):
    ens = (data.dropna(subset=[col]).groupby([col, "lag_bin"])["msd_um2"]
               .agg(["mean", "sem", "count"]).reset_index())
    alphas = {}
    for r in order:
        e = ens[(ens[col] == r) & (ens["count"] >= MIN_TRACKS_PER_LAG)].sort_values("lag_bin")
        if e.empty:
            continue
        # errorbar returns (line, caplines, barlinecols); fade only the bars/caps
        _, caps, bars = ax.errorbar(e["lag_bin"], e["mean"], yerr=e["sem"], marker="o",
                                    color=colors[r], capsize=3, lw=1.6, label=r)
        for artist in (*caps, *bars):
            artist.set_alpha(0.35)        # faded error bars; markers/line stay solid
        fit = e[(e["lag_bin"] <= FIT_MAX_LAG_MIN) & (e["mean"] > 0)]
        if len(fit) >= 3:                            # slope of log(MSD) vs log(tau) = alpha
            a, b = np.polyfit(np.log(fit["lag_bin"]), np.log(fit["mean"]), 1)
            alphas[r] = a
            xx = np.array([fit["lag_bin"].min(), fit["lag_bin"].max()], float)
            ax.plot(xx, np.exp(b) * xx**a, ls="--", lw=1, color=colors[r])
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("lag time \u03c4 (min)")
    ax.set_title(title, fontsize=11)
    ax.legend(title="region")
    return alphas

# sharex + sharey so the two panels sit on IDENTICAL log scales -> comparable
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
aL = draw_msd(axL, msd, "region",     REGIONS_MSD, MOT_COLORS,
              "Three-group gradient\n(does the side matter?)")
aR = draw_msd(axR, msd, "bin_region", ORDER2,      BIN_COLORS,
              f"Binary collapse\n({IN_LABEL} vs {OUT_LABEL})")
axL.set_ylabel("MSD  \u27e8\u0394r\u00b2\u27e9  (\u00b5m\u00b2)")   # shared y -> label left only

# ---- clamp x to where data actually exists (kills the empty left half) ----
# sharex=True -> setting axL also sets axR
lag_lo = msd["lag_bin"][msd["lag_bin"] > 0].min()
lag_hi = msd["lag_bin"].max()
axL.set_xlim(lag_lo * 0.8, lag_hi * 1.2)

fig.suptitle("Mean squared displacement, two groupings (time-averaged, ensemble)", fontsize=12)
fig.tight_layout(); plt.show()

# ---- alpha read-out for both groupings -----------------------------------
def tag(a): return "confined/sub-diffusive" if a < 0.85 else \
                   ("directed/super-diffusive" if a > 1.15 else "~diffusive")
print("Fitted log-log slope \u03b1 (short lags):")
print("  three-group:")
for r in REGIONS_MSD:
    if r in aL: print(f"     {r:>9}: \u03b1 = {aL[r]:0.2f}   ({tag(aL[r])})")
print("  binary:")
for r in ORDER2:
    if r in aR: print(f"     {r:>12}: \u03b1 = {aR[r]:0.2f}   ({tag(aR[r])})")
print("\n   \u03b1<1 confined (plateau)   \u03b1\u22481 diffusive   \u03b1>1 directed (\u03b1\u21922 ballistic)")

# Where this leaves you, and what to reach for next

**What the notebook now answers**
- **Q1 (recruitment):** Cells 3b–6 — a visual QC overview (region balance, unlinked spots, manual-curation evenness), counts per region over real time per image, and a **paired within-fish comparison** whose 95% CI is built on the *per-fish differences* (`injured − uninjured`), not on each region separately — the honest unit for a paired design.
- **Q2 (parked vs roaming):** Cells 7–11b — the per-track motility table, per-image **core-boundary crossing** counts, per-track speed/confinement, and step-speed distributions; the motility, step-speed and **MSD** cells each show the full `uninjured → penumbra → core` gradient *beside* its binary `core vs rest` collapse, so you can see whether the collapse is faithful; the trajectory maps then show it cell-by-cell, with a territory ribbon per cell.

**What the side-by-side panels revealed (worth keeping)**
- The **penumbra is its own state**, not a twin of uninjured: it's the *fastest* group in step speed, while the core is slowest — so the binary collapse is defensible for the "is the core slower?" question but *lossy* for the biology, because it averages the penumbra peak away.
- The MSD α tells a complementary story (uninjured & penumbra both directed, core ~diffusive), where the same binary collapse *is* fairly faithful — the penumbra resembles uninjured in **directedness** even while differing in **raw speed**.

**Caveats carried through (don't lose these in a methods write-up)**
1. **Counts aren't area-normalised** — part of any injured/uninjured difference could be region size. Export region area (µm²) from Step 04 and divide.
2. **Pooling ignores fish-level grouping** — every "pooled" comparison treats tracks (and steps) as independent. The paired CI in Cell 6 is exploratory, not a hypothesis test; for a real statistic use a mixed-effects model with fish as a random effect.
3. **Track region labels use majority assignment** — a boundary-crossing cell is filed wholly under whichever region it spent most frames in, which contaminates per-region track summaries. The step-level views (step-speed, core-crossing) sidestep this by labelling each *step*; MSD can't, since it needs the whole trajectory.
4. **Unlinked detections** (~11% here) count toward Q1 but not Q2 — `COUNT_ONLY_TRACKED` in Cell 4 lets you test sensitivity (and Cell 5 now echoes which mode is active).
5. **Manual spots** are valid for counts/position, invalid for morphology — `is_manual` / `fraction_manual` keep them traceable, and Cell 3b shows curation isn't spread evenly across fish.
6. **Few tracks per region** — MSD α and the per-track jitter medians are indicative, not conclusive; trust the curve/distribution shape over the fitted number, and the territory ribbons are a qualitative coverage cue, not a measured µm² area. Revisit once you have the full 5–6 series.

**Natural next analyses** (each is a small extension of a table you already built):
- **Distance-to-injury over time** per track — needs the injury boundary geometry exported from Step 04 (the open README TODO: line vs polygon).
- **Convergence / flux across the boundary** — the per-image core-crossing cell already counts crossing *events* from `region_fine` changes; net (signed) flux is one step further.
- **Behavioural subtypes** — cluster tracks on `[speed, confinement, directedness, frac_core]` from `per_track`.

> One small documentation nit I noticed while checking: the repo **README prose** still says "injured = 1, uninjured = 2", but your current `04_draw_injury.ijm` actually writes **uninjured = 1, injured = 2, core = 3** — which is what this notebook (correctly) assumes, and what your sample data contains. Worth fixing the README line and the older `data_schema.md` so a future-you isn't misled.